# Percobaan 1 — Ensemble sementara (14 model, 4 arsitektur)

Notebook **terpisah**, bukan bagian dari alur NB03/NB04 resmi — dibuat untuk mengumpulkan satu
submission awal SEBELUM semua 20 model (4 arsitektur x 5 fold) selesai.

**Status checkpoint saat ini (dari `Training_Output` di Drive):**
- ConvNeXt V2 — 5/5 fold selesai
- EfficientNetV2-M — 5/5 fold selesai
- Swin Transformer V2 — baru 2/5 fold (fold0, fold1)
- MaxViT — baru 2/5 fold (fold0, fold1)

Total **14 model** dipakai (5+5+2+2).

**Perbedaan sengaja dari NB03/NB04 asli (dicatat, bukan bug):**
1. **Tidak ada bobot ensemble hasil optimasi OOF** — file OOF (`oof_{arch}_fold{k}.csv`) belum
   lengkap untuk SwinV2/MaxViT (baru 2 fold), jadi belum bisa jalankan optimasi Nelder-Mead seperti
   NB03 asli. Sebagai gantinya dipakai **rata-rata sama rata antar 4 arsitektur** (bobot 1/4 masing-
   masing) — pilihan wajar untuk submission awal, BUKAN pengganti permanen NB03.
2. **Tidak ada kalibrasi logit-adjustment** — itu juga butuh OOF lengkap. Delta kalibrasi di sini 0
   (tidak ada penyesuaian).
3. Dalam satu arsitektur, fold yang tersedia tetap dirata-rata sama rata (2 fold untuk SwinV2/MaxViT,
   5 fold untuk ConvNeXt/EfficientNet) — bagian ini SAMA persis dengan logika NB04 asli.

Begitu SwinV2 & MaxViT selesai sampai 5 fold dan OOF lengkap terkumpul, submission berikutnya WAJIB
pakai NB03+NB04 asli (bobot teroptimasi + kalibrasi), bukan notebook percobaan ini lagi.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import timm.data
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 3
print("device:", DEVICE, "| torch:", torch.__version__, "| timm:", timm.__version__)


## Config path

Sama seperti NB01/NB04 versi Colab: root Drive, folder data mentah `BDC2026/`, dan folder
checkpoint manual `Training_Output/{NamaFolderArsitektur}/fold{k}/best.ckpt`.


In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive/Big Data Challenge - Satria Data 2026")
TEST_ROOT = DRIVE_ROOT / "BDC2026" / "test"
TEMPLATE_PATH = DRIVE_ROOT / "BDC2026" / "submission.csv"
CKPT_ROOT = DRIVE_ROOT / "Training_Output"
TEST_IMAGE_EXT = "jpg"

OUTPUT_ROOT = DRIVE_ROOT / "Preprocessing_Output" / "submissions"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SUBMISSION_TAG = "KepepetDeadline"  # cocokkan dengan nama file lokal yang sudah ada; ganti kalau perlu

for _check_path in [TEST_ROOT, TEMPLATE_PATH, CKPT_ROOT]:
    if not _check_path.exists():
        raise FileNotFoundError(f"{_check_path} tidak ketemu -- cek Drive ter-mount dan path benar.")

# ARCH_REGISTRY: nama folder Drive HARUS persis sama (spasi/kapitalisasi) dengan yang ada di
# Training_Output, dan available_folds HARUS cocok dengan fold yang benar-benar sudah py punya
# best.ckpt -- kalau ada fold yang belum selesai tapi ikut dimasukkan di sini, load-nya akan gagal.
ARCH_REGISTRY = {
    "convnextv2_base": dict(
        timm_tag="convnextv2_base.fcmae_ft_in22k_in1k_384", drop_path_rate=0.3,
        drive_folder="ConvNeXt V2", available_folds=[0, 1, 2, 3, 4],
    ),
    "swinv2_base_window12to24": dict(
        timm_tag="swinv2_base_window12to24_192to384.ms_in22k_ft_in1k", drop_path_rate=0.3,
        drive_folder="Swin Transformer V2", available_folds=[0, 1],
    ),
    "maxvit_base_tf": dict(
        timm_tag="maxvit_base_tf_384.in21k_ft_in1k", drop_path_rate=0.3,
        drive_folder="MaxViT", available_folds=[0, 1],
    ),
    "tf_efficientnetv2_m": dict(
        timm_tag="tf_efficientnetv2_m.in21k_ft_in1k", drop_path_rate=0.2,
        drive_folder="EfficientNetV2-M", available_folds=[0, 1, 2, 3, 4],
    ),
}

for _arch, _spec in ARCH_REGISTRY.items():
    for _k in _spec["available_folds"]:
        _p = CKPT_ROOT / _spec["drive_folder"] / f"fold{_k}" / "best.ckpt"
        if not _p.exists():
            raise FileNotFoundError(f"{_p} tidak ketemu -- cek nama folder/fold di ARCH_REGISTRY.")

total_models = sum(len(s["available_folds"]) for s in ARCH_REGISTRY.values())
print(f"Total model yang akan dipakai: {total_models}")


## Dataset test, transform, TTA, pembuatan model

Byte-identical secara logika dengan NB04 asli (PRD §7.3, §7.8, §11) -- cuma sumber checkpoint dan
bobot ensemble yang beda, bukan cara inferensinya.


In [ ]:
class TestDataset(Dataset):
    def __init__(self, image_ids: np.ndarray, transform):
        self.image_ids = image_ids.astype(np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = int(self.image_ids[idx])
        path = f"{TEST_ROOT}/{image_id}.{TEST_IMAGE_EXT}"
        with Image.open(path) as im:
            im = im.convert("RGB")
            tensor = self.transform(im)
        return tensor, image_id


def build_eval_transform(img_size, mean, std):
    resize_short_side = round(img_size / 0.95)
    return T.Compose([
        T.Resize(resize_short_side, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(img_size),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])


TTA_SPEC_VERSION = "identity+hflip-softmax-mean-v1"  # sama persis dengan skeleton 02x & NB04


@torch.no_grad()
def tta_predict_probs(model, images: torch.Tensor) -> torch.Tensor:
    images = images.to(DEVICE, non_blocking=True)
    probs_identity = F.softmax(model(images), dim=1)
    probs_flip = F.softmax(model(torch.flip(images, dims=[3])), dim=1)
    return (probs_identity + probs_flip) / 2.0


def create_inference_model(timm_tag: str, drop_path_rate: float, img_size: int):
    kwargs = dict(pretrained=False, num_classes=NUM_CLASSES, drop_path_rate=drop_path_rate, img_size=img_size)
    try:
        model = timm.create_model(timm_tag, **kwargs)
    except TypeError:
        kwargs.pop("img_size")
        model = timm.create_model(timm_tag, **kwargs)
    return model


def resolve_norm_stats(model):
    data_cfg = timm.data.resolve_model_data_config(model)
    return list(data_cfg["mean"]), list(data_cfg["std"])


## Enumerasi gambar test dari template resmi


In [ ]:
template_df = pd.read_csv(TEMPLATE_PATH)
template_ids = template_df["id"].values
assert len(template_ids) == 1458, f"seharusnya 1458 baris template, dapat {len(template_ids)}"
print(f"template: {len(template_ids)} baris, rentang id [{template_ids.min()}, {template_ids.max()}]")


## Inferensi TTA per-arsitektur, per-fold yang tersedia -> rata-rata fold -> probabilitas arsitektur


In [ ]:
arch_probs = {}  # arch -> array probabilitas (n_test, 3), dirata-rata dari fold yang TERSEDIA saja

for arch, spec in ARCH_REGISTRY.items():
    fold_prob_sum = np.zeros((len(template_ids), NUM_CLASSES), dtype=np.float64)
    id_to_row = {int(v): i for i, v in enumerate(template_ids)}

    for k in spec["available_folds"]:
        ckpt_path = CKPT_ROOT / spec["drive_folder"] / f"fold{k}" / "best.ckpt"
        best = torch.load(ckpt_path, map_location="cpu")
        img_size = best["img_size"]

        model = create_inference_model(spec["timm_tag"], spec["drop_path_rate"], img_size).to(DEVICE)
        model.load_state_dict(best["weights"])
        model.eval()
        mean, std = resolve_norm_stats(model)

        eval_tf = build_eval_transform(img_size, mean, std)
        loader = DataLoader(TestDataset(template_ids, eval_tf), batch_size=32, shuffle=False,
                             drop_last=False, num_workers=2, pin_memory=True)

        for images, image_ids in tqdm(loader, desc=f"{arch} fold{k}", leave=False):
            probs = tta_predict_probs(model, images).cpu().numpy()
            for image_id, p in zip(image_ids.numpy(), probs):
                fold_prob_sum[id_to_row[int(image_id)]] += p

        del model
        torch.cuda.empty_cache()

    arch_probs[arch] = fold_prob_sum / len(spec["available_folds"])
    print(f"{arch}: inferensi selesai ({len(spec['available_folds'])} fold) untuk {len(template_ids)} gambar test")


## Gabungan antar arsitektur — rata-rata sama rata (BUKAN bobot teroptimasi OOF)

Beda dari NB03/NB04 asli: di sini `final_weights[arch] = 1 / jumlah_arsitektur` untuk semua, dan
tidak ada delta kalibrasi (0). Ini sengaja sederhana untuk submission awal -- begitu OOF 4
arsitektur lengkap (semua 5 fold selesai), ulangi lewat NB03 asli untuk bobot & kalibrasi yang benar.


In [ ]:
n_archs = len(arch_probs)
ens_probs = np.zeros((len(template_ids), NUM_CLASSES), dtype=np.float64)
for arch, probs in arch_probs.items():
    ens_probs += probs / n_archs

final_preds = ens_probs.argmax(axis=1)


## Validation gate (SEBELUM menulis file submission)

Tepat 1458 baris; set id dan urutan sama dengan template; `predicted` di {0,1,2}; tidak ada NaN —
sama persis aturan NB04 asli.


In [ ]:
submission_df = pd.DataFrame({"id": template_ids, "predicted": final_preds})

assert len(submission_df) == 1458
assert np.array_equal(submission_df["id"].values, template_ids), "urutan baris menyimpang dari template — jangan pernah sort ulang"
assert submission_df["predicted"].isin([0, 1, 2]).all()
assert not submission_df.isna().any().any()
print("Validation gate LOLOS.")

output_path = OUTPUT_ROOT / f"submission_{SUBMISSION_TAG}.csv"
submission_df.to_csv(output_path, index=False)
print(f"Menulis {output_path}")

test_dist = submission_df["predicted"].value_counts(normalize=True).sort_index().to_dict()
print("distribusi prediksi di test set:", {k: round(v, 4) for k, v in test_dist.items()})


## Pengingat

Ini submission dari **14/20 model** dengan bobot rata-rata sama, bukan hasil ensemble teroptimasi
penuh. Kalau dipakai sebagai submission "asuransi" pertama (PRD §11), sisakan jatah submission untuk
versi final setelah SwinV2 & MaxViT selesai 5 fold dan dijalankan lewat NB03+NB04 resmi.
